In [ ]:
import pandas as pd
import warnings
import numpy as np
from sklearn.svm import SVC
from rdkit import Chem
import warnings
from data_processing import dataset_process
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# 定义文件路径
dataset_path = "./SAM_data_PCE_15%_0721.csv"
hl_path = "./SAM_data_PCE_15%_0721_with_HOMO_LUMO.csv"

In [ ]:
dataset = pd.read_csv(dataset_path)
dataset_homo_lumo = pd.read_csv(hl_path)

In [ ]:
# 指定所需的特征
features_to_include = ['ecfp', 'substrate', 'pvk_ions', 'pvk_bandgap'] # 'substrate', 'pvk_ions', 'pvk_bandgap'

# 调用 dataset_process 函数
processed_dataset = dataset_process(dataset, features_to_include)

In [ ]:
processed_dataset

In [ ]:
#按骨架核心分类
df_copy = processed_dataset.copy()

df_1 = pd.DataFrame(columns=df_copy.columns)
df_2 = pd.DataFrame(columns=df_copy.columns)
df_3 = pd.DataFrame(columns=df_copy.columns)
df_4 = pd.DataFrame(columns=df_copy.columns)
df_5 = pd.DataFrame(columns=df_copy.columns)
df_6 = pd.DataFrame(columns=df_copy.columns)
df_7 = pd.DataFrame(columns=df_copy.columns)

# 链状
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        if num_rings == 0:
            df_1 = pd.concat([df_1, pd.DataFrame([row])], ignore_index=True)
            df_copy.drop(index, inplace=True)

# 萘酰胺
patt_1 = Chem.MolFromSmiles('O=C1NC(=O)C2=CC=CC3=C2C1=CC=C3')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_1):
        df_2 = pd.concat([df_2, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 三苯胺
patt_2 = Chem.MolFromSmiles('C1=CC=C(C=C1)N(C1=CC=CC=C1)C1=CC=CC=C1')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_2):
        df_3 = pd.concat([df_3, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 咔唑
patt_3 = Chem.MolFromSmiles('N1C2=C(C=CC=C2)C2=C1C=CC=C2')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_3):
        df_4 = pd.concat([df_4, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 联噻吩
patt_4 = Chem.MolFromSmiles('S1C=CC=C1C1=CC=CS1')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_4):
        df_5 = pd.concat([df_5, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 单环
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        if num_rings == 1:
            df_6 = pd.concat([df_6, pd.DataFrame([row])], ignore_index=True)
            df_copy.drop(index, inplace=True)

# 其他
df_7 = df_copy.copy()

In [ ]:
# 训练和测试划分函数
def split_train_test(dfs):
    df_test = pd.DataFrame()
    df_train = pd.DataFrame()
    for df_sub in dfs:
        if len(df_sub) <= 10:
            test_indices = np.random.choice(df_sub.index, size=1, replace=False)
        elif 10 < len(df_sub) <= 20:
            test_indices = np.random.choice(df_sub.index, size=2, replace=False)
        elif 20 < len(df_sub) <= 30:
            test_indices = np.random.choice(df_sub.index, size=3, replace=False)
        elif 30 < len(df_sub) <= 40:
            test_indices = np.random.choice(df_sub.index, size=4, replace=False)
        elif 40 < len(df_sub) <= 50:
            test_indices = np.random.choice(df_sub.index, size=5, replace=False)
        elif 50 < len(df_sub):
            test_indices = np.random.choice(df_sub.index, size=6, replace=False)
        else:
            test_indices = []

        df_test = pd.concat([df_test, df_sub.loc[test_indices]], ignore_index=False)
        train_indices = df_sub.index.difference(test_indices)
        df_train = pd.concat([df_train, df_sub.loc[train_indices]], ignore_index=False)

    return df_train, df_test

# 抽样多次进行训练和测试，并计算平均混淆矩阵
def train_and_evaluate_multiple_samples(dfs, best_params, n_samples=200, seed=0):
    """
    重复 n_samples 次：
    - 按 split_train_test 规则划分 train/test
    - 训练 SVC
    - 在 test 上计算 confusion matrix
    返回：平均混淆矩阵 mean_cm（float）
    """
    if seed is not None:
        np.random.seed(seed)

    cm_list = []
    labels_all = [0, 1, 2]  # 你的三分类标签

    for _ in range(n_samples):
        df_train, df_test = split_train_test(dfs)

        X_train = df_train.iloc[:, 1:-1].astype(float)
        y_train = df_train['label'].astype(int)

        X_test = df_test.iloc[:, 1:-1].astype(float)
        y_test = df_test['label'].astype(int)

        model = SVC(
            C=best_params['C'],
            kernel=best_params['kernel'],
            gamma=best_params['gamma'],
            probability=True
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        cm = confusion_matrix(y_test, y_pred, labels=labels_all)
        cm_list.append(cm)

    mean_cm = np.mean(np.stack(cm_list, axis=0), axis=0)
    return mean_cm

def plot_confusion_matrix_percentage(mean_cm,
                                     class_names,
                                     font="Arial",
                                     font_size=22,
                                     cmap="GnBu",
                                     figsize=(8, 7),
                                     dpi=600):

    # ===== 归一化为行百分比 =====
    cm_percentage = mean_cm.astype(float)
    row_sums = cm_percentage.sum(axis=1, keepdims=True)
    cm_percentage = np.divide(cm_percentage, row_sums, where=row_sums!=0) * 100

    # ===== 全局字体 =====
    plt.rcParams.update({
        "font.family": font,
        "font.size": font_size,
        "axes.labelweight": "normal",
        "axes.titleweight": "normal"
    })

    fig = plt.figure(figsize=figsize, dpi=dpi)
    ax = fig.add_axes([0.18, 0.15, 0.75, 0.75])

    # ===== 画 heatmap =====
    sns.heatmap(
        cm_percentage,
        annot=True,
        fmt=".1f",
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        linewidths=1.5,
        linecolor="black",
        cbar=True,
        vmin=0,
        vmax=70,
        annot_kws={"size": font_size-2}
    )

    # ===== 在每个格子后面加 % =====
    for text in ax.texts:
        text.set_text(f"{float(text.get_text()):.1f}%")
        text.set_fontfamily(font)

    # ===== 坐标轴设置 =====
    ax.set_xlabel("Predicted label", labelpad=10)
    ax.set_ylabel("True label", labelpad=10)
    ax.set_xticklabels(class_names, rotation=10, ha='right')
    ax.set_yticklabels(class_names, rotation=80, va='center')
    ax.tick_params(axis='both', which='major', length=6, width=1.5)

    # ===== 边框加粗 =====
    for spine in ax.spines.values():
        spine.set_linewidth(2)

    # ===== colorbar 设置 =====
    cbar = ax.collections[0].colorbar
    # 给刻度加 %
    ticks = cbar.get_ticks()
    cbar.set_ticks(ticks)
    cbar.set_ticklabels([f"{int(t)}%" for t in ticks])
    cbar.ax.tick_params(labelsize=font_size-4)

    plt.show()

class_names = ['Low PCE', 'Medium PCE', 'High PCE']
dfs = [df_1, df_2, df_3, df_4, df_5, df_6]
best_params = { 'C': 50, 'kernel': 'linear', 'gamma': 0.0001 }    
mean_cm = train_and_evaluate_multiple_samples(dfs, best_params, n_samples=200)
plot_confusion_matrix_percentage(mean_cm, class_names)

In [ ]:
from sklearn.metrics import roc_curve, auc
from matplotlib.ticker import AutoMinorLocator
def plot_roc_curve_mean(dfs, best_params, n_samples=200, seed=0):
    """
    论文级 ROC 汇总：
    - 重复 n_samples 次 split + train + test
    - 每类计算 ROC，并插值到统一 FPR 网格
    - 输出 mean ROC，并可视化 ±1 std 阴影
    - 输出 AUC mean ± std
    """
    # ===== 全局字体统一 Arial =====
    plt.rcParams.update({
        'font.family': 'Arial',
        'font.size': 24,
        'font.weight': 'normal',
        'axes.labelweight': 'normal',
        'axes.titleweight': 'normal',
        'xtick.labelsize': 22,
        'ytick.labelsize': 22,
        'legend.fontsize': 18,
        'mathtext.fontset': 'custom',
        'mathtext.rm': 'Arial',
        'mathtext.it': 'Arial',
        'mathtext.bf': 'Arial'
    })

    # 固定随机性（保证可复现，但不影响“平均”的统计意义）
    if seed is not None:
        np.random.seed(seed)

    class_names = ['Low PCE', 'Medium PCE', 'High PCE']
    colors = ['#5E72FF', '#FB6F6F', '#008C5F']
    n_classes = len(class_names)

    # 统一的 FPR 网格（用于插值求平均）
    mean_fpr = np.linspace(0.0, 1.0, 200)

    # 存每次的插值 TPR & AUC
    tpr_list = {i: [] for i in range(n_classes)}
    auc_list = {i: [] for i in range(n_classes)}

    for _ in range(n_samples):
        df_train, df_test = split_train_test(dfs)

        df_train_X = df_train.iloc[:, 1:-1].astype(float)
        df_test_X  = df_test.iloc[:, 1:-1].astype(float)
        df_train_y = df_train['label'].astype(int)
        df_test_y  = df_test['label'].astype(int)

        model = SVC(
            C=best_params['C'],
            kernel=best_params['kernel'],
            gamma=best_params['gamma'],
            probability=True
        )
        model.fit(df_train_X, df_train_y)
        y_score = model.predict_proba(df_test_X)

        for i in range(n_classes):
            # one-vs-rest ROC
            fpr, tpr, _ = roc_curve(df_test_y == i, y_score[:, i])
            roc_auc = auc(fpr, tpr)

            # 插值到统一网格（保证可平均）
            tpr_interp = np.interp(mean_fpr, fpr, tpr)
            tpr_interp[0] = 0.0  # 规范起点

            tpr_list[i].append(tpr_interp)
            auc_list[i].append(roc_auc)

    # ===== 计算 mean/std =====
    mean_tpr = {}
    std_tpr = {}
    mean_auc = {}
    std_auc = {}

    for i in range(n_classes):
        tprs = np.vstack(tpr_list[i])  # shape: (n_samples, len(mean_fpr))
        mean_tpr[i] = tprs.mean(axis=0)
        std_tpr[i] = tprs.std(axis=0)

        aucs = np.array(auc_list[i])
        mean_auc[i] = aucs.mean()
        std_auc[i] = aucs.std()

    # ================= 绘图 =================
    fig = plt.figure(figsize=(7.5, 7.5), dpi=600)
    ax = fig.add_axes([0.16, 0.16, 0.78, 0.78])

    for i in range(n_classes):
        ax.plot(
            mean_fpr, mean_tpr[i],
            lw=2.5,
            color=colors[i],
            label=f'{class_names[i]} (AUC = {mean_auc[i]:.2f} ± {std_auc[i]:.2f})'
        )
        # ±1 std 阴影
        lower = np.clip(mean_tpr[i] - std_tpr[i], 0, 1)
        upper = np.clip(mean_tpr[i] + std_tpr[i], 0, 1)
        ax.fill_between(mean_fpr, lower, upper, color=colors[i], alpha=0.15, linewidth=0)

    # 对角线
    ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=2)

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.02)

    ax.set_xlabel('False Positive Rate (FPR)')
    ax.set_ylabel('True Positive Rate (TPR)')

    # 刻度（主/次）
    ax.tick_params(axis='both', which='major', direction='out', length=8, width=2)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(axis='both', which='minor', direction='out', length=5, width=2)

    # 边框
    for spine in ax.spines.values():
        spine.set_linewidth(2)

    # 图例
    legend = ax.legend(
        loc='lower right',
        fontsize=16,
        frameon=False,
        handlelength=2.5
)

    plt.show()

    # 同时返回数值，方便你写 SI/正文
    return mean_auc, std_auc

mean_auc, std_auc = plot_roc_curve_mean(dfs, best_params, n_samples=200, seed=0)
print(mean_auc, std_auc)